# San Antonio Bicycle High Injury Network

This notebook uses the City's official 22 Bicycle High Injury Network corridors. It compares the City's 2019–2023 study period with the current CRIS reporting window, 2024 through Sept. 1, 2026. It does not invent a new hotspot rule.

In [ ]:
from pathlib import Path
import requests
import pandas as pd
import geopandas as gpd

ROOT = Path.cwd()
RAW = ROOT / 'data' / 'raw'
BOUNDARIES = ROOT / 'data' / 'boundaries'
OUT = ROOT / 'outputs'
BOUNDARIES.mkdir(exist_ok=True)
OUT.mkdir(exist_ok=True)

In [ ]:
# Load CRIS and keep one row per qualifying bicyclist.
raw = pd.read_csv(RAW / 'myexport_final.csv', skiprows=12, low_memory=False)
severity = {'K - FATAL INJURY', 'A - SUSPECTED SERIOUS INJURY'}
target = raw[(raw['Person Type'] == '3 - PEDALCYCLIST') & raw['Person Injury Severity'].isin(severity)].copy()
target['year'] = pd.to_numeric(target['Crash Year'], errors='coerce')
target['latitude'] = pd.to_numeric(target['Latitude'], errors='coerce')
target['longitude'] = pd.to_numeric(target['Longitude'], errors='coerce')
target['death'] = (target['Person Injury Severity'] == 'K - FATAL INJURY').astype(int)
target['serious_injury'] = (target['Person Injury Severity'] == 'A - SUSPECTED SERIOUS INJURY').astype(int)
sa = target[(target['City'] == 'SAN ANTONIO') & target['year'].between(2019, 2026)].copy()
crashes = (sa.groupby('Crash ID', as_index=False)
    .agg(year=('year', 'first'), latitude=('latitude', 'first'), longitude=('longitude', 'first'),
         deaths=('death', 'sum'), serious_injuries=('serious_injury', 'sum')))
geo = crashes.dropna(subset=['latitude', 'longitude'])
points = gpd.GeoDataFrame(geo, geometry=gpd.points_from_xy(geo['longitude'], geo['latitude']), crs=4326)
print('San Antonio qualifying crashes:', crashes['Crash ID'].nunique())

In [ ]:
# Download and use the official Bicycle HIN layer.
hin_url = ('https://services.arcgis.com/g1fRTDLeMgspWrYp/arcgis/rest/services/'
           'SS4A_HIN_Dashboard_Data/FeatureServer/1/query?'
           'where=1%3D1&outFields=*&returnGeometry=true&outSR=4326&f=geojson')
hin_file = BOUNDARIES / 'bicycle_hin_corridors.geojson'
response = requests.get(hin_url, timeout=120)
response.raise_for_status()
hin_file.write_bytes(response.content)
hin = gpd.read_file(hin_file).to_crs(4326).rename(columns={'Name': 'corridor'})
print('Official HIN corridors:', len(hin))

# The buffer assigns crashes to the City's existing corridor lines.
# It does not create new corridors.
hin_buffer = hin.to_crs(2278).copy()
hin_buffer['geometry'] = hin_buffer.geometry.buffer(150)
matches = gpd.sjoin(points.to_crs(2278), hin_buffer[['bicycle_hin_id', 'corridor', 'Miles', 'geometry']],
                    how='inner', predicate='within')
matches = matches.drop_duplicates(['Crash ID', 'bicycle_hin_id'])

def summarize(frame):
    return (frame.groupby(['bicycle_hin_id', 'corridor', 'Miles'], as_index=False)
        .agg(crashes=('Crash ID', 'nunique'), deaths=('deaths', 'sum'),
             serious_injuries=('serious_injuries', 'sum'), first_year=('year', 'min'),
             last_year=('year', 'max')))

baseline = summarize(matches[matches['year'].between(2019, 2023)])
current = summarize(matches[matches['year'].between(2024, 2026)])
baseline.to_csv(OUT / 'official_bicycle_hin_2019_2023.csv', index=False)
current.to_csv(OUT / 'official_bicycle_hin_2024_2026.csv', index=False)

matched_crash_ids = matches['Crash ID'].drop_duplicates()
outside = crashes[~crashes['Crash ID'].isin(matched_crash_ids)].copy()
outside.to_csv(OUT / 'qualifying_crashes_outside_official_hin.csv', index=False)
print('Qualifying crashes outside official HIN:', len(outside))
display(baseline.sort_values(['crashes', 'deaths'], ascending=False))
display(current.sort_values(['crashes', 'deaths'], ascending=False))

## How to use this

The two corridor CSVs show whether the City's official corridors remain active. `qualifying_crashes_outside_official_hin.csv` identifies severe/fatal bicyclist crashes outside those corridors. Because the City has not published its original analytical model, this notebook does not claim to recreate the exact ranking algorithm.